# 📊 Análise Exploratória - Detector de Anomalias em Transações

Este notebook realiza uma análise exploratória (EDA) dos dados de transações financeiras
e demonstra o funcionamento dos algoritmos de detecção de anomalias.

## Seções
1. Carregamento e inspeção inicial dos dados
2. Estatísticas descritivas
3. Visualizações (distribuições, séries temporais, correlações)
4. Aplicação dos detectores
5. Avaliação das métricas
6. Dashboard interativo

## 1. Setup e imports

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)
print(f'Pandas: {pd.__version__}')
print(f'NumPy:  {np.__version__}')

## 2. Carregamento dos dados

Tenta carregar do MySQL; se falhar, cai para o CSV de fallback.

In [ ]:
from src.data_loader import carregar_transacoes_mysql, carregar_transacoes_csv

try:
    df = carregar_transacoes_mysql()
    if len(df) == 0:
        raise RuntimeError('vazio')
    print(f'✅ Carregado do MySQL: {len(df)} linhas')
except Exception as e:
    print(f'⚠️ MySQL indisponível ({e}), usando CSV...')
    df = carregar_transacoes_csv('../data/raw/transacoes.csv')
    print(f'✅ Carregado do CSV: {len(df)} linhas')

df.head()

## 3. Estatísticas descritivas

In [ ]:
print('Shape:', df.shape)
print('\nTipos:')
print(df.dtypes)
print('\nNulos por coluna:')
print(df.isna().sum())
df.describe(include='all').T

In [ ]:
# Distribuição por tipo de transação
df['tipo_transacao'].value_counts().plot(kind='bar', figsize=(10, 4), title='Tipos de Transação')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Distribuição por local
df['local'].value_counts().head(15).plot(kind='barh', figsize=(10, 6), title='Top 15 Locais')
plt.tight_layout()
plt.show()

## 4. Distribuição de valores

Valores de transações tipicamente seguem distribuição log-normal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df['valor'].hist(bins=50, ax=axes[0], color='steelblue')
axes[0].set_title('Valor (escala original)')
axes[0].set_xlabel('Valor (R$)')

np.log1p(df['valor']).hist(bins=50, ax=axes[1], color='darkgreen')
axes[1].set_title('log(1 + Valor)')
axes[1].set_xlabel('log(Valor)')
plt.tight_layout()
plt.show()

## 5. Pré-processamento + Feature Engineering

In [ ]:
from src.preprocessor import TransactionPreprocessor

pre = TransactionPreprocessor()
df_feat, X = pre.fit_transform(df)
print(f'Shape da matriz de features: {X.shape}')
print(f'Features: {pre.feature_columns}')
df_feat.head()

## 6. Heatmap de correlações

In [ ]:
corr = df_feat[pre.feature_columns].corr()
plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Correlação entre Features')
plt.tight_layout()
plt.show()

## 7. Detecção de anomalias

In [ ]:
from src.anomaly_detector import detectar_anomalias_combinado

r_if, r_db = detectar_anomalias_combinado(X, contamination=0.03)
print(f'Isolation Forest detectou {int(r_if.labels.sum())} anomalias')
print(f'DBSCAN+ZScore detectou    {int(r_db.labels.sum())} anomalias')

## 8. Avaliação (se houver labels verdadeiros)

In [ ]:
from src.evaluator import avaliar, imprimir_relatorio

if 'is_anomaly_true' in df_feat.columns:
    y_true = df_feat['is_anomaly_true'].values
    print(imprimir_relatorio(y_true, r_if.labels, 'Isolation Forest'))
    print(imprimir_relatorio(y_true, r_db.labels, 'DBSCAN + Z-Score'))
    m_if = avaliar(y_true, r_if.labels, r_if.scores, 'isolation_forest')
    m_db = avaliar(y_true, r_db.labels, r_db.scores, 'dbscan_zscore')
    print(f'\nIsolation Forest → F1={m_if.f1:.3f}, AUC={m_if.auc_roc:.3f}')
    print(f'DBSCAN+ZScore   → F1={m_db.f1:.3f}, AUC={m_db.auc_roc:.3f}')
else:
    print('Sem labels verdadeiros disponíveis.')

## 9. Visualização das anomalias em 2D

In [ ]:
df_feat['is_anomaly_pred'] = r_if.labels

fig = px.scatter(
    df_feat,
    x='valor_log',
    y='desvio_valor_cliente',
    color='is_anomaly_pred',
    color_discrete_map={0: 'steelblue', 1: 'crimson'},
    hover_data=['cliente_id', 'valor', 'local', 'tipo_transacao'],
    title='Anomalias detectadas (Isolation Forest)',
    opacity=0.6,
)
fig.show()

## 10. Dashboard completo

In [ ]:
from src.visualizations import gerar_todos_graficos

df_viz = df_feat.copy()
df_viz['is_anomaly'] = r_if.labels
paths = gerar_todos_graficos(df_viz, output_dir='../data/processed')
print('Arquivos gerados:')
for nome, caminho in paths.items():
    print(f'  - {nome}: {caminho}')

## ✅ Conclusão

- O pipeline carrega transações do MySQL (ou CSV de fallback)
- Gera features temporais, de cliente e categóricas
- Aplica Isolation Forest (principal) e DBSCAN+Z-Score (secundário)
- Avalia com F1, Precision, Recall e AUC-ROC
- Produz visualizações estáticas (matplotlib/seaborn) e interativas (Plotly)